# Faruq-v3 S2 — STB1-guided WAV-L1 YOLO26, seed 42
Frozen Stage-A screen for the lightweight student: `RGB -> WAV-L1 -> native YOLO26n`, trained with a **training-only frozen STB1 cross-head teacher**.

**Required private Kaggle inputs:**
1. `faruq-v3-experiment-core-v1`
2. `faruq-v3-stb1-teacher-addon-v1`

This notebook runs **seed 42 only** on grouped development validation. S3 AF2 robustness training, seeds 123/2026, hyperparameter retuning, and locked test are explicitly blocked.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
core=sorted(INPUT.rglob('af2_spectral_kaggle_manifest.json'))
teacher=sorted(INPUT.rglob('stb_guided_teacher_manifest.json'))
if len(core)!=1: raise FileNotFoundError(f'Harus ada tepat satu core manifest; ditemukan {core}')
if len(teacher)!=1: raise FileNotFoundError(f'Harus ada tepat satu teacher manifest; ditemukan {teacher}')
print('INPUT PREFLIGHT PASS')
print('CORE:',core[0])
print('TEACHER:',teacher[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,torch
os.chdir(WORK)
REPO=WORK/'coffee-bean-detection'; BRANCH='agent/stb-guided-robust-wav-yolo'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All.')


In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
from coffee_detector.experiments.prepare_stb_guided_kaggle import prepare_stb_guided_teacher_addon
DATA,ARTIFACTS,CORE_CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
TEACHER,TEACHER_CONTRACT=prepare_stb_guided_teacher_addon(INPUT)
assert CORE_CONTRACT['test_images_accessed'] is False and TEACHER_CONTRACT['test_images_accessed'] is False
D0=Path(ARTIFACTS['D0_seed42_best.pt']); GROUPED=DATA/'faruq_grouped_summary.json'
if not D0.is_file() or not GROUPED.is_file() or not TEACHER.is_file():
    raise FileNotFoundError(f'Missing core artifact: D0={D0} GROUPED={GROUPED} TEACHER={TEACHER}')
print('INPUT CONTRACTS PASS')
print('DATA:',DATA)
print('D0:',D0)
print('STB1 TEACHER:',TEACHER)
print('GROUPED:',GROUPED)


In [ ]:
# Unit tests + architecture audit are hard blockers before any GPU training.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_stb_guided.py'],cwd=REPO,check=True)
OUT=WORK/'stb-guided-s2-seed42-v1'; OUT.mkdir(exist_ok=True)
STATIC=OUT/'static_audit.json'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_guided',
     '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--d0-checkpoint',str(D0),
     '--teacher-checkpoint',str(TEACHER),'--output-root',str(OUT),'--stage','static','--seed','42']
subprocess.run(cmd,cwd=REPO,check=True)
audit=json.loads(STATIC.read_text(encoding='utf-8'))
print(json.dumps(audit,indent=2))
if audit['decision']!='PASS' or audit['test_opened'] is not False:
    raise RuntimeError('STOP: S2 static audit gagal. Jangan training.')
print('STATIC AUTHORIZATION PASS')
print('STUDENT PARAMS:',audit['parameters']['stb_guided_student'])
print('WAV-L1 PARAMS:',audit['parameters']['wav_l1_reference'])
print('AF2 TRAINING-VIEW PARAMS:',audit['parameters']['af2_training_view'])


In [ ]:
# Train exactly one frozen S2 arm: seed 42 only.
RESULT=OUT/'val_reports'/'stb_guided_s2_seed42_decision.json'
LOG=OUT/'S2_STB_CROSSKD_seed42.log'
if not RESULT.is_file():
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_guided',
         '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--d0-checkpoint',str(D0),
         '--teacher-checkpoint',str(TEACHER),'--output-root',str(OUT),
         '--stage','train','--seed','42','--device','0','--authorize-training']
    print('START S2 STB1-guided WAV-L1 seed42',flush=True)
    with LOG.open('a',encoding='utf-8') as stream:
        p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    previous=None
    while p.poll() is None:
        csv=OUT/'S2_STB_CROSSKD_seed42'/'results.csv'
        epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=previous: print(f'S2: {epoch}/50 epoch',flush=True); previous=epoch
        time.sleep(120)
    if p.returncode:
        tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-220:]) if LOG.is_file() else '<no log>'
        raise RuntimeError(f'S2 gagal, returncode={p.returncode}\n{tail}')
else:
    print('REUSE COMPLETE RESULT:',RESULT)
if not RESULT.is_file(): raise RuntimeError(f'Hasil tidak ditemukan: {RESULT}')
result=json.loads(RESULT.read_text(encoding='utf-8'))
assert result['evaluation_split']=='val' and result['test_opened'] is False and result['test_images_accessed'] is False
assert result['s3_training_authorized'] is False
print('\n=== STB-GUIDED WAV-L1 S2 SEED42 SCREEN ===')
for name,metrics in result['models'].items(): print(name,metrics)
print('\nDELTA VS WAV-L1:',result['comparison']['delta_vs_WAV_L1'])
print('RETENTION:',result['comparison']['retention'])
print('ADVANCEMENT:',result['comparison']['advancement'])
print('DESCRIPTIVE DELTA VS STB1 TEACHER:',result['comparison']['descriptive_delta_vs_STB1_teacher'])
print('\nDECISION:',result['decision'])
print('NEXT:',result['next_action'])
print('DEPLOYMENT:',result['student_deployment'])
archive=Path(shutil.make_archive(str(WORK/'stb-guided-s2-seed42-output'),'zip',OUT))
print('\nFINAL ZIP:',archive)
print('RESULT:',RESULT)
print('STOP HERE. Jangan jalankan seed 123/2026, S3 AF2, tuning, atau locked test sebelum hasil S2 direview.')
